# Get to Know a Dataset: LAFC-Evict

**Dataset:** LAFC-Evict: Learning-Augmented Cache Eviction Dataset (v1.0)  
**Current AWS S3 release:** `s3://lafc-evict-open-data/v1.0/`  
**Region:** `us-west-2`  
**Historical release:** v0.3 remains available at the bucket root for reproducibility.

LAFC-Evict is a derived, tabular research dataset of counterfactual supervision labels for learning-augmented cache-eviction research. v1.0 contains five trace families: `cloudphysics`, `metacdn`, `metakv`, `twemcache`, and `wiki2018`. The release includes 277,995,072 candidate rows, 2,363,286 decision-view rows, and a 1,000,000-row pairwise sample.

This notebook uses public anonymous S3 access and reads only small metadata plus one small Parquet partition, so it does not require downloading the full ~3 GB release.

## Dataset organization

The current release is versioned under `v1.0/`:

```text
s3://lafc-evict-open-data/v1.0/
  README.md
  data/candidate_rows/split=.../trace_family=.../capacity=.../horizon=.../candidate_rows.parquet
  data/decision_view/decision_view.parquet
  data/pairwise_sample/pairwise_sample.parquet
  metadata/release_manifest.json
  metadata/checksums.sha256
```

The historical v0.3 files remain at `s3://lafc-evict-open-data/` for reproducibility. New work should start from `s3://lafc-evict-open-data/v1.0/`.

## Load release metadata

This first example fetches only the release manifest over HTTPS. It is inexpensive and works without AWS credentials.

In [ ]:
import json
from urllib.request import urlopen

BASE_HTTPS = "https://lafc-evict-open-data.s3.us-west-2.amazonaws.com/v1.0"
manifest_url = f"{BASE_HTTPS}/metadata/release_manifest.json"

with urlopen(manifest_url) as response:
    manifest = json.load(response)

summary = {
    "version": manifest["version"],
    "candidate_rows": manifest["candidate_row_count"],
    "decision_rows": manifest["decision_row_count"],
    "pairwise_rows": manifest["pairwise_sample_row_count"],
    "families": manifest["selected_families"],
    "excluded_families": manifest["excluded_families"],
}
summary

## Read one small candidate-row Parquet object

The full candidate-row tree is partitioned by split, family, capacity, and horizon. The example below reads one small Wiki2018 validation partition directly from public S3 over HTTPS with PyArrow. You can swap the URL for another partition without changing the code shape.

In [ ]:
from io import BytesIO
from urllib.request import urlopen

import pyarrow.parquet as pq
import pandas as pd

candidate_url = (
    f"{BASE_HTTPS}/data/candidate_rows/"
    "split=val/trace_family=wiki2018/capacity=32/horizon=16/candidate_rows.parquet"
)

with urlopen(candidate_url) as response:
    parquet_bytes = response.read()

candidate_table = pq.read_table(BytesIO(parquet_bytes))
print("downloaded bytes:", len(parquet_bytes))
print("rows:", candidate_table.num_rows)
print("columns:", candidate_table.column_names)

cols = [
    "trace_family", "split", "capacity", "horizon",
    "y_loss", "y_value", "candidate_lru_score", "candidate_predictor_score",
]
sample_df = candidate_table.select(cols).to_pandas()
sample_df.head()

## A tiny exploratory check

This is not a statistical benchmark; it is just a quick sanity check over one partition. For real analysis, aggregate across the relevant families, splits, capacities, and horizons.

In [ ]:
summary_df = (
    sample_df
    .groupby(["trace_family", "split", "capacity", "horizon"], as_index=False)
    .agg(
        rows=("y_loss", "size"),
        mean_y_loss=("y_loss", "mean"),
        mean_lru_score=("candidate_lru_score", "mean"),
        mean_predictor_score=("candidate_predictor_score", "mean"),
    )
)
summary_df

## Pairwise and decision views

v1.0 also publishes:

- `data/decision_view/decision_view.parquet` with 2,363,286 decision rows.
- `data/pairwise_sample/pairwise_sample.parquet` with 1,000,000 candidate-pair rows.

The pairwise labels are represented by three mutually exclusive boolean columns: `label_a_better`, `label_b_better`, and `is_tie`. The validated v1.0 label counts are 60,673, 61,065, and 878,262 respectively.

Those files are larger than the single candidate partition above, so this introductory notebook describes them rather than downloading them by default.

## Provenance, privacy, and scope

v1.0 includes five cleared families: `cloudphysics`, `metacdn`, `metakv`, `twemcache`, and `wiki2018`. The `citibike` and `brightkite` families are excluded. LAFC-Evict is a derived dataset; it does not redistribute raw upstream traces, raw Wikimedia pageview dumps, or raw page titles. Wiki2018 identifiers are deterministically pseudonymized in the public release.

Use the checksum manifest at `metadata/checksums.sha256` when mirroring or auditing objects. Do not rely on S3 ETags as SHA-256 checksums, especially for multipart objects.

## Citation

```bibtex
@dataset{vahidi_lafc_evict_v1_2026,
  author    = {Vahidi, Soroush},
  title     = {LAFC-Evict: Learning-Augmented Cache Eviction Dataset (v1.0)},
  year      = {2026},
  version   = {1.0},
  publisher = {AWS Open Data},
  url       = {s3://lafc-evict-open-data/v1.0/}
}
```

Please also cite and comply with the relevant upstream raw-data sources for any analysis that depends on their provenance.